In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/nba_games.csv')


df['date'] = pd.to_datetime(df['date'])


df = df.sort_values(['team', 'date']).reset_index(drop=True)


print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDate range:", df['date'].min(), "to", df['date'].max())
print("\nUnique teams:", df['team'].nunique())
print("\nSample rows:")
df[['team', 'date', 'opponent', 'is_home', 'team_score', 'result']].head(10)

Shape: (35995, 31)

Columns: ['SEASON_ID', 'TEAM_ID', 'team', 'TEAM_NAME', 'GAME_ID', 'date', 'matchup', 'result_str', 'MIN', 'team_score', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'result', 'is_home', 'opponent']

Date range: 2012-10-05 00:00:00 to 2025-06-22 00:00:00

Unique teams: 99

Sample rows:


,team,date,opponent,is_home,team_score,result
0,ADL,2018-10-05,UTA,0,99,0
1,ADL,2019-10-05,UTA,0,81,0
2,ADL,2022-10-02,PHX,0,134,1
3,ADL,2022-10-06,OKC,0,98,0
4,ALB,2012-10-06,DAL,1,84,0
5,ALB,2014-10-08,SAS,1,94,1
6,ATL,2012-10-07,MIA,1,92,1
7,ATL,2012-10-10,SAS,0,99,0
8,ATL,2012-10-14,MEM,0,102,0
9,ATL,2012-10-16,IND,0,98,0


In [9]:

grp = df.groupby('team')


df['roll_pts_scored'] = grp['team_score'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)


df['roll_plus_minus'] = grp['PLUS_MINUS'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)


df['roll_win_rate'] = grp['result'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)


df['roll_fg_pct'] = grp['FG_PCT'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)


df['days_rest'] = grp['date'].diff().dt.days.fillna(7).clip(1, 7)


df['streak'] = grp['result'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).sum()
)

print("Features added!")
print(df[['team', 'date', 'roll_pts_scored', 'roll_win_rate', 
          'days_rest', 'streak']].head(10))

Features added!
  team       date  roll_pts_scored  roll_win_rate  days_rest  streak
0  ADL 2018-10-05              NaN            NaN        7.0     NaN
1  ADL 2019-10-05        99.000000       0.000000        7.0     0.0
2  ADL 2022-10-02        90.000000       0.000000        7.0     0.0
3  ADL 2022-10-06       104.666667       0.333333        4.0     1.0
4  ALB 2012-10-06              NaN            NaN        7.0     NaN
5  ALB 2014-10-08        84.000000       0.000000        7.0     0.0
6  ATL 2012-10-07              NaN            NaN        7.0     NaN
7  ATL 2012-10-10        92.000000       1.000000        3.0     1.0
8  ATL 2012-10-14        95.500000       0.500000        4.0     1.0
9  ATL 2012-10-16        97.666667       0.333333        2.0     1.0


In [10]:
df.to_csv('data/processed/nba_features.csv', index=False)
print(f"Saved {len(df)} rows with {len(df.columns)} columns")

Saved 35995 rows with 37 columns


In [11]:
NBA_TEAMS = [
    'ATL', 'BOS', 'BKN', 'CHI', 'CHA', 'CLE', 'DAL', 'DEN', 'DET',
    'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN',
    'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS',
    'TOR', 'UTA', 'WAS'
]

df = df[df['team'].isin(NBA_TEAMS)].copy()
df = df[df['SEASON_ID'].astype(str).str.startswith('2')].copy()
df = df.reset_index(drop=True)

print(f"After filtering: {len(df)} games, {df['team'].nunique()} teams")
print("Teams:", sorted(df['team'].unique()))

After filtering: 31254 games, 30 teams
Teams: ['ATL', 'BKN', 'BOS', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']
